In [ ]:
import cv2
import mediapipe as mp
import numpy as np

# MediaPipe 손 인식 모듈 불러오기
mp_hands = mp.solutions.hands
# 한 손만 인식하도록 설정 (신뢰도 0.7)
hands = mp_hands.Hands(max_num_hands=1, min_detection_confidence=0.7)
# 화면에 뼈대를 그려주는 도구
mp_drawing = mp.solutions.drawing_utils

def get_normalized_landmarks(hand_landmarks):
    # 21개 관절의 3D 좌표를 담을 빈 배열 생성
    landmarks = np.zeros((21, 3))
    
    # 추출된 관절 좌표를 배열에 차례대로 채워 넣기
    for i, lm in enumerate(hand_landmarks.landmark):
        landmarks[i] = [lm.x, lm.y, lm.z]
    
    # 0번(손목) 좌표를 기준점(원점)으로 설정
    base = landmarks[0]
    # 모든 좌표에서 손목 좌표를 빼서 상대 위치로 변환
    landmarks = landmarks - base
    
    # 크기에 상관없이 모양만 보기 위해 최대값 구하기
    max_val = np.max(np.abs(landmarks))
    # 최대값이 0보다 크면 전체를 최대값으로 나누어 크기 정규화
    if max_val > 0:
        landmarks = landmarks / max_val
        
    # 정규화가 완료된 좌표 반환
    return landmarks

def calculate_similarity(lm1, lm2):
    # 두 제스처 간의 유클리디안 거리 평균 오차 계산 (작을수록 똑같음)
    return np.mean(np.square(lm1 - lm2))

# 웹캠 실행
cap = cv2.VideoCapture(0)

# 등록할 TARGET_COUNT개의 제스처 데이터를 저장할 리스트
registered_data = []
# 목표 등록 개수 설정
TARGET_COUNT = int(input("등록할 제스처 개수를 입력하세요 (예: 10): "))
# 물리적 버튼(인식 활성화) 상태 변수
recognition_active = False

print("========================================")
print(f"먼저 {TARGET_COUNT}개의 제스처를 순서대로 등록합니다.")
print("========================================")

# 카메라가 열려있는 동안 무한 반복
while cap.isOpened():
    # 카메라에서 사진(프레임) 1장 읽어오기
    ret, frame = cap.read()
    # 읽어오기 실패하면 반복문 종료
    if not ret:
        break
        
    # 거울처럼 보이게 좌우 반전
    frame = cv2.flip(frame, 1)
    # OpenCV의 BGR 색상을 MediaPipe용 RGB 색상으로 변환
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    # 손 랜드마크 분석 실행
    results = hands.process(rgb_frame)
    # 현재 화면의 제스처 정보를 담을 변수 초기화
    current_gesture = None
    
    # 화면에 손이 감지되었다면
    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            # 손가락 관절과 뼈대를 화면에 그리기
            mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)
            # 현재 손동작을 정규화된 좌표로 변환하여 저장
            current_gesture = get_normalized_landmarks(hand_landmarks)

    # 현재까지 등록된 제스처 개수 확인
    current_count = len(registered_data)
    
    # [상태 1] TARGET_COUNT개를 다 채우기 전 (등록 모드)
    if current_count < TARGET_COUNT:
        # 화면 좌측 상단에 현재 등록 진행 상황 표시
        text = f"Registering: {current_count}/{TARGET_COUNT}. Press 'Enter'"
        cv2.putText(frame, text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
    
    # [상태 2] TARGET_COUNT개를 모두 등록한 후 (실전 인식 모드)
    else:
        # 버튼이 켜진 상태라면 (인식 중)
        if recognition_active:
            status_text = "Button ON : Recognizing..."
            color = (0, 255, 0) # 초록색 텍스트
        # 버튼이 꺼진 상태라면 (대기 중)
        else:
            status_text = "Button OFF: Press 'b' to start"
            color = (0, 0, 255) # 빨간색 텍스트
            
        # 버튼 켜짐/꺼짐 상태를 화면에 출력
        cv2.putText(frame, status_text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
        
        # 버튼이 켜져있고, 화면에 손이 보일 때만 판별 시작
        if recognition_active and current_gesture is not None:
            # 가장 작은 오차를 찾기 위해 초기값을 무한대로 설정
            min_error = float('inf')
            # 일치하는 제스처의 명령어를 담을 변수
            matched_command = ""
            
            # 저장해둔 10개의 제스처를 하나씩 꺼내서 비교
            for data in registered_data:
                # 저장된 제스처와 현재 제스처의 오차 계산
                error = calculate_similarity(data['gesture'], current_gesture)
                # 만약 지금까지 본 오차보다 더 작다면(더 비슷하다면)
                if error < min_error:
                    # 최소 오차 기록 갱신
                    min_error = error
                    # 해당 명령어(텍스트) 기억해두기
                    matched_command = data['command']
            
            # 가장 비슷한 제스처의 오차가 0.02보다 작을 때만 (확실할 때만) 실행
            if min_error < 0.02: 
                # 해당 명령어를 화면 중앙에 파란색으로 크게 띄우기
                cv2.putText(frame, matched_command, (50, 200), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (255, 0, 0), 3)

    # 최종 완성된 프레임을 화면에 띄우기
    cv2.imshow('Gesture Control System', frame)
    
    # 1ms 동안 키보드 입력 대기
    key = cv2.waitKey(1) & 0xFF
    
    # [동작 1] 엔터키(13) 누름 + TARGET_COUNT개 미만 등록 + 손 인식됨
    if key == 13 and current_count < TARGET_COUNT and current_gesture is not None:
        print(f"\n✅ [{current_count + 1}/{TARGET_COUNT}] 번째 제스처 캡처 완료!")
        # 터미널 창을 통해 실행할 텍스트 명령어 입력받기
        cmd = input("▶ 이 제스처의 기능을 입력하세요 (예: 에어컨 켜기): ")
        # 입력받은 정보(좌표, 텍스트)를 묶어서 리스트에 저장
        registered_data.append({'gesture': current_gesture, 'command': cmd})
        
        # 방금 TARGET_COUNT개째 등록을 마쳤다면 완료 안내 메시지 출력
        if len(registered_data) == TARGET_COUNT:
            print(f"\n {TARGET_COUNT}개의 제스처가 모두 등록되었습니다!")
            print("이제 영상 창을 클릭하고 키보드 'b'를 눌러 인식을 시작하세요.")
            
    # [동작 2] 'b'키 누름 + TARGET_COUNT개 등록 완료 상태 (버튼 토글)
    elif key == ord('b') and current_count == TARGET_COUNT:
        # 버튼 상태 반대로 뒤집기 (True <-> False)
        recognition_active = not recognition_active
        if recognition_active:
            print("🔘 버튼 켜짐: 제스처 인식을 시작합니다.")
        else:
            print("🔘 버튼 꺼짐: 제스처 인식을 중지합니다.")
            
    # [동작 3] 'q'키를 누르면 프로그램 강제 종료
    elif key == ord('q'):
        break

# 사용이 끝난 카메라 장치 해제
cap.release()
# 열려있는 모든 OpenCV 창 닫기
cv2.destroyAllWindows()

먼저 5개의 제스처를 순서대로 등록합니다.


c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '



✅ [1/5] 번째 제스처 캡처 완료!

✅ [2/5] 번째 제스처 캡처 완료!

✅ [3/5] 번째 제스처 캡처 완료!

✅ [4/5] 번째 제스처 캡처 완료!

✅ [5/5] 번째 제스처 캡처 완료!

 5개의 제스처가 모두 등록되었습니다!
이제 영상 창을 클릭하고 키보드 'b'를 눌러 인식을 시작하세요.
🔘 버튼 켜짐: 제스처 인식을 시작합니다.
